# Matching experiment analysis

Observations for a selected matching experiment are loaded from MongoDB, and node and relation matching quality is analyzed.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from pymongo import MongoClient

EXPERIMENT_NAME = "matching-gpt5-6-luna"
COLLECTION_NAME = "matching_eval"
MONGODB_URI = os.getenv(
    "MONGODB_URI",
    "mongodb://127.0.0.1:27017/llm_uml_evaluator?replicaSet=rs0",
)

plt.style.use("seaborn-v0_8-whitegrid")

## Data loading

The URI is read from the `MONGODB_URI` environment variable. JupyterLab is started with `uv run --env-file .env --group eval jupyter lab`.

In [ ]:
client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5_000)
try:
    collection = client.get_default_database()[COLLECTION_NAME]
    documents = list(
        collection.find(
            {"experiment_name": EXPERIMENT_NAME},
            {
                "experiment_name": 1,
                "sample": 1,
                "mutation": 1,
                "nodes": 1,
                "relations": 1,
            },
        )
    )
finally:
    client.close()

if not documents:
    raise ValueError(f"No observations found for {EXPERIMENT_NAME!r}")

len(documents)

## Metric calculation

A match is a `(reference_uid, candidate_uid)` pair. The formulas are applied separately to node matches and relation matches:

- `TP`: a returned pair that is also expected;
- `FP`: a returned pair that is not expected;
- `FN`: an expected pair that was not returned.

```text
Precision = TP / (TP + FP)
Recall    = TP / (TP + FN)
F1        = 2 × Precision × Recall / (Precision + Recall)
          = 2TP / (2TP + FP + FN)
```

The final, algebraically equivalent F1 formula is used. A metric with a zero denominator remains undefined (`NaN`) and is excluded from the mean.

In [ ]:
df = pd.json_normalize(documents).rename(
    columns={
        "nodes.true_positive": "nodes_tp",
        "nodes.false_positive": "nodes_fp",
        "nodes.false_negative": "nodes_fn",
        "relations.true_positive": "relations_tp",
        "relations.false_positive": "relations_fp",
        "relations.false_negative": "relations_fn",
    }
)

required_columns = {
    "sample",
    "mutation",
    "nodes_tp",
    "nodes_fp",
    "nodes_fn",
    "relations_tp",
    "relations_fp",
    "relations_fn",
}
if missing := required_columns.difference(df.columns):
    raise ValueError(f"Missing columns: {sorted(missing)}")

for component in ("nodes", "relations"):
    tp = df[f"{component}_tp"]
    fp = df[f"{component}_fp"]
    fn = df[f"{component}_fn"]
    precision_denominator = tp + fp
    recall_denominator = tp + fn
    f1_denominator = 2 * tp + fp + fn
    df[f"{component}_precision"] = tp / precision_denominator.where(
        precision_denominator.ne(0)
    )
    df[f"{component}_recall"] = tp / recall_denominator.where(
        recall_denominator.ne(0)
    )
    df[f"{component}_f1"] = 2 * tp / f1_denominator.where(
        f1_denominator.ne(0)
    )

df.head()

## Coverage check

The table shows the number of repetitions for each `sample × mutation` pair.

In [ ]:
coverage = (
    df.groupby(["sample", "mutation"])
    .size()
    .unstack(fill_value=0)
)
display(coverage)
display(
    df.groupby("mutation")[
        ["nodes_f1", "relations_f1"]
    ].agg(["mean", "std", "count"]).round(3)
)

## Quality by mutation

Points show mean F1; vertical bars show one standard deviation across observations.

In [ ]:
f1_by_mutation = df.groupby("mutation")[["nodes_f1", "relations_f1"]].agg(
    ["mean", "std"]
)

fig, ax = plt.subplots(figsize=(10, 5))
for component, label in (("nodes", "Nodes"), ("relations", "Relations")):
    ax.errorbar(
        f1_by_mutation.index,
        f1_by_mutation[(f"{component}_f1", "mean")],
        yerr=f1_by_mutation[(f"{component}_f1", "std")],
        marker="o",
        capsize=4,
        label=label,
    )
ax.set(
    title=f"Matching F1 by mutation — {EXPERIMENT_NAME}",
    xlabel="Mutation",
    ylabel="F1",
    ylim=(0, 1.05),
)
ax.legend()
plt.show()

## Error profile

Mean number of false (`FP`) and missed (`FN`) matches per observation.

In [ ]:
errors = df.groupby("mutation")[
    ["nodes_fp", "nodes_fn", "relations_fp", "relations_fn"]
].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, component, title in (
    (axes[0], "nodes", "Nodes"),
    (axes[1], "relations", "Relations"),
):
    ax.bar(errors.index, errors[f"{component}_fp"], label="FP")
    ax.bar(
        errors.index,
        errors[f"{component}_fn"],
        bottom=errors[f"{component}_fp"],
        label="FN",
    )
    ax.set(title=title, xlabel="Mutation")
axes[0].set_ylabel("Mean errors per observation")
axes[1].legend()
fig.suptitle(f"Matching errors — {EXPERIMENT_NAME}")
fig.tight_layout()
plt.show()

## Problematic samples and mutations

F1 is averaged across repetitions in the heatmap. A quality drop can then be associated with a particular diagram or mutation type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, component, title in (
    (axes[0], "nodes", "Nodes F1"),
    (axes[1], "relations", "Relations F1"),
):
    matrix = df.pivot_table(
        index="sample",
        columns="mutation",
        values=f"{component}_f1",
        aggfunc="mean",
    )
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set(
        title=title,
        xlabel="Mutation",
        ylabel="Sample",
        xticks=range(len(matrix.columns)),
        xticklabels=matrix.columns,
        yticks=range(len(matrix.index)),
        yticklabels=matrix.index,
    )
    for row in range(len(matrix.index)):
        for column in range(len(matrix.columns)):
            value = matrix.iloc[row, column]
            label = "n/a" if pd.isna(value) else f"{value:.2f}"
            ax.text(column, row, label, ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=axes, label="Mean F1")
plt.show()